# AxiomZero EIGE v21.0 — Quickstart Notebook

**Election Integrity Governance Engine**  
Version 21.0.0 | Phase 1-B Complete | 449 tests passing

This notebook walks through the complete EIGE pipeline on a small synthetic election:

1. Understanding the core tamper-detection invariants
2. Ingesting ballots through a county node
3. Demonstrating tamper detection
4. Running the HolographicScreen normalisation layer
5. State-wide aggregation and Holon Zero certificate
6. Federal blind audit verification
7. Reading the Public Trust Report

---
*Theory & scientific direction: ThomasCory Walker-Pearson*  
*Code architecture & implementation: GitHub Copilot (AI)*

## 0. Setup

Run from the repository root directory.  Make sure you have installed the requirements:
```bash
pip install -r EIGE/requirements.txt
```

In [ ]:
import sys, os
# Add repo root to path so 'EIGE.src' is importable
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))  # adjust if needed
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import math
print(f'Python {sys.version}')

## 1. Core Tamper-Detection Invariants

EIGE uses two constants as its tamper-detection anchor:

- **`K_CS = 74`** — the accumulator seed for the path-dependent rolling hash
- **`φ₀ = π/4`** — the equilibrium scalar that a legitimate ballot sequence always converges to

These come from the Unitary Manifold physics framework, but their **operational validity
requires only their mathematical properties**, not the cosmological theory.  See
[`src/constants_engineering.py`](../src/constants_engineering.py) for the physics-free equivalents.

In [ ]:
from EIGE.src.constants import K_CS, PHI_0, PHI_TOLERANCE, SHARD_COUNT, COUNTY_COUNT

print(f'K_CS (accumulator seed)     = {K_CS}  (= 5² + 7²)')
print(f'PHI_0 (equilibrium scalar)  = {PHI_0:.16f}  (= π/4)')
print(f'PHI_TOLERANCE (hard limit)  = {PHI_TOLERANCE}')
print(f'SHARD_COUNT                 = {SHARD_COUNT}  shards per county node')
print(f'COUNTY_COUNT                = {COUNTY_COUNT}  Washington State counties')

## 2. The Chern-Simons Rolling Hash

The hash accumulates ballot integers in a **path-dependent, non-commutative** way:

```
s₀ = K_CS = 74
s_{n+1} = ((s_n × K_CS + b_n) XOR (s_n >> 7)) mod M63
```

The key property: `hash([a, b, c]) ≠ hash([b, a, c])`.  Any reordering is immediately detectable.

In [ ]:
from EIGE.src.chern_simon_hash import ChernSimonChain

# Legitimate sequence: [100, 200, 300]
chain_legit = ChernSimonChain()
for b in [100, 200, 300]:
    chain_legit.accumulate(b)

# Tampered sequence: same numbers, different order
chain_tampered = ChernSimonChain()
for b in [200, 100, 300]:   # reordered
    chain_tampered.accumulate(b)

print(f'Legitimate hash state : {chain_legit.state}')
print(f'Tampered  hash state  : {chain_tampered.state}')
print(f'Are they equal?       : {chain_legit.state == chain_tampered.state}')
print('\nEven a single ballot reorder produces a completely different hash.')

## 3. County Node Ingestion

Each county runs a `CountyNode` that:
- Accepts integer ballot vectors
- Maintains a sharded hash chain
- Computes `φ_eff` for closure validation
- Handles network partitions gracefully

In [ ]:
from EIGE.src.county_node import CountyNode
from EIGE.src.metric_closure import ClosureStatus

county = CountyNode('WA-047', 'King County')

# Ingest 200 synthetic ballots (5 races, 3 candidates each)
import random
random.seed(42)
for _ in range(200):
    county.ingest_ballot([random.randint(0, 2) for _ in range(5)])

result = county.validate_closure()
print(f'Ballot count  : {county.ballot_count()}')
print(f'φ_eff         : {result.phi_eff:.16f}')
print(f'φ₀            : {PHI_0:.16f}')
print(f'|φ_eff − φ₀|  : {abs(result.phi_eff - PHI_0):.2e}')
print(f'Closure status: {result.status.name}')

## 4. Tamper Detection — Live Demonstration

We now simulate two ballot-stuffing scenarios and watch the closure status change.

In [ ]:
from EIGE.src.metric_closure import MetricClosure

# Build two county nodes with the same legitimate ballots
legit_county = CountyNode('WA-047-LEGIT', 'King County (Legitimate)')
tampered_county = CountyNode('WA-047-TAMPER', 'King County (Tampered)')

ballots = [[random.randint(0, 2) for _ in range(5)] for _ in range(500)]
for b in ballots:
    legit_county.ingest_ballot(b)
    tampered_county.ingest_ballot(b)

# Now stuff 50 extra ballots into the tampered county
for _ in range(50):
    tampered_county.ingest_ballot([2, 2, 2, 2, 2])  # all-candidate-2 stuffed ballots

legit_result = legit_county.validate_closure()
tampered_result = tampered_county.validate_closure()

print('=== Legitimate County ===')
print(f'  Ballots: {legit_county.ballot_count()}')
print(f'  Status : {legit_result.status.name}')
print()
print('=== Tampered County (50 stuffed ballots) ===')
print(f'  Ballots: {tampered_county.ballot_count()}')
print(f'  Status : {tampered_result.status.name}')
print()
print('The tampered county diverges from φ₀ — the sequence was disrupted.')

## 5. HolographicScreen — Normalising Real-World Scanner Input

Real ballot scanners produce float confidence scores, not clean integers.
The `HolographicScreen` normalises these with explicit, auditable rules —
no machine learning, no adaptive algorithms.

In [ ]:
from EIGE.src.holographic_screen import HolographicScreen, WriteInRegistry, AdmissibilityError
from EIGE.src.holographic_screen import RawBallotRecord

registry = WriteInRegistry()
registry.register('alice johnson', 0)
registry.register('bob chen', 1)

screen = HolographicScreen(write_in_registry=registry)

# High-confidence float ballot — passes through
high_conf = [0.98, 0.95, 0.92, 0.88, 0.99]
result_high = screen.screen(high_conf)
print(f'High-confidence ballot → {result_high.selection_vector}  | action: {result_high.action}')

# Low-confidence ballot — routed to human adjudicator
low_conf = [0.45, 0.50, 0.30, 0.55, 0.40]
try:
    result_low = screen.screen(low_conf)
except AdmissibilityError as e:
    print(f'Low-confidence ballot  → AdmissibilityError (human queue): {e}')

print(f'\nNormalisation log entries: {len(screen.normalisation_log)}')

## 6. State Mesh — Aggregation and Holon Zero Certificate

The `StateMesh` aggregates results from all county nodes and emits
a Holon Zero Certificate — a zero-knowledge commitment to the invariants
that the federal tier can verify without seeing any raw ballot data.

In [ ]:
from EIGE.src.state_mesh import StateMesh

# Build a small multi-county election
nodes = [CountyNode(f'WA-{i:03d}', f'County {i}') for i in range(5)]
for node in nodes:
    for _ in range(300):
        node.ingest_ballot([random.randint(0, 2) for _ in range(5)])

mesh = StateMesh(nodes, jurisdiction_id='WA-STATE-DEMO')
entry = mesh.compute_braid_sync()

print(f'Jurisdiction          : {entry.jurisdiction_id}')
print(f'Counties verified     : {entry.counties_verified}/{len(nodes)}')
print(f'Aggregate ballot count: {entry.aggregate_ballot_count:,}')
print(f'Aggregate closure     : {entry.aggregate_closure.status.name}')

cert = entry.holon_zero_cert
if cert:
    print(f'Holon Zero Cert       : phi_verified={cert.phi_verified}, k_cs_verified={cert.k_cs_verified}')
else:
    print('Holon Zero Cert       : not emitted (state not fully STABLE)')

## 7. Federal Blind Audit

The `FederalAuditor` receives only OSCAL 1.5.0 Holon Zero Certificates.
Any attempt to access raw ballot data raises `RawDataAccessAttempt` — by
design, not by policy.

In [ ]:
from EIGE.src.federal_auditor import FederalAuditor, RawDataAccessAttempt

auditor = FederalAuditor()

if cert:
    verdict = auditor.validate_certificate(cert)
    print(f'Audit verdict   : {verdict.verdict.name}')
    print(f'φ verified      : {verdict.phi_verified}')
    print(f'k_CS verified   : {verdict.k_cs_verified}')

# Attempt to access raw ballot data — must be blocked
print()
try:
    _ = auditor.query_raw_votes()   # type: ignore
except (RawDataAccessAttempt, AttributeError) as e:
    print(f'Raw data query BLOCKED: {type(e).__name__}')
    print('(This is the intended behaviour — federal tier has no ballot access)')

## 8. Public Trust Report

The final output layer translates the 5D geometry into plain English
suitable for election directors, courts, and journalists.

The `PublicTrustReport` contains **zero physics vocabulary**.

In [ ]:
from EIGE.src.public_trust_index import PublicTrustIndexBuilder

builder = PublicTrustIndexBuilder()
report = builder.build_from_ledger_entry(
    entry=entry,
    jurisdiction_name='Washington State (Demo)',
)

print(f'STATUS: {report.status}')
print()
print('Plain-English Summary:')
print('-' * 70)
print(report.plain_english_summary)
print()
print('Statistical Equivalent:')
print('-' * 70)
print(report.statistical_equivalent)

## Next Steps

- **Full test suite:** `python -m pytest EIGE/tests/ -v`  (449 tests)
- **End-to-end demo:** `python EIGE/run_demo.py`
- **Architecture docs:** [`EIGE/ARCHITECTURE.md`](../ARCHITECTURE.md)
- **Full technical book:** [`EIGE/BOOK.md`](../BOOK.md)
- **NIST compliance mapping:** [`EIGE/COMPLIANCE.md`](../COMPLIANCE.md)
- **Physics-free constants:** [`EIGE/src/constants_engineering.py`](../src/constants_engineering.py)

---
*Theory, framework, and scientific direction: **ThomasCory Walker-Pearson**.*  
*Code architecture, test suites, document engineering, and synthesis: **GitHub Copilot** (AI).*